# Efficient-SAM3-Finetuning — GPU Test Runner

One-click runner for GPU-gated tests (currently: `spec/peft-qlora`).

**Prereqs:**
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. In Colab Secrets (left sidebar 🔑), add:
   - `HF_TOKEN` — Hugging Face access token with read access to gated `facebook/sam3.1`.
   - `GH_TOKEN` — GitHub fine-grained personal access token with **Contents: Read** on this repo. **Required only if the repo is private**; can be omitted for public repos.
3. Edit `BRANCH` in the next cell if testing something other than `main`.
4. Runtime → Run all.

In [ ]:
BRANCH = "main"
REPO = "NguyenJus/Efficient-SAM3-Finetuning"

In [ ]:
# Cell 1: Environment guard.
import sys

import torch

assert "google.colab" in sys.modules, "This notebook is intended for Google Colab."
assert torch.cuda.is_available(), (
    "No CUDA device. Runtime → Change runtime type → T4 GPU or better."
)
cc = torch.cuda.get_device_capability()
assert cc >= (
    7,
    5,
), f"CUDA compute capability {cc} is < 7.5. bitsandbytes 4-bit requires Turing or later."
print(f"GPU OK: {torch.cuda.get_device_name(0)} (CC {cc[0]}.{cc[1]})")

In [ ]:
# Cell 2: Clone + checkout.
import os
import subprocess

from google.colab import userdata

# GH_TOKEN is optional: required for private repos, ignored for public ones.
try:
    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

clone_url = (
    f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
)

if not os.path.isdir("Efficient-SAM3-Finetuning"):
    subprocess.run(["git", "clone", clone_url], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "fetch", "--all"], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "checkout", BRANCH], check=True)
os.chdir("Efficient-SAM3-Finetuning")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# Cell 3: Install dev + qlora + tensorboard extras.
# %pip (not !pip) guarantees the install lands in the kernel's Python rather
# than a sibling system Python on Colab — otherwise pytest later imports a
# different env. Drop -q so install errors surface here, not in cell 6.
# Upgrade huggingface_hub up front so the new `hf` CLI is available and the
# version-check prompt does not block the `hf download` cell.
%pip install -e ".[qlora,dev,tensorboard]" "huggingface_hub>=1.15"
!python -c "import esam3; print('esam3 OK:', esam3.__file__)"

In [ ]:
# Cell 4: HF auth (token in Colab Secrets).
import os

from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "HF_TOKEN missing from Colab Secrets. See the prereqs cell."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token  # huggingface-cli reads this name too

In [ ]:
# Cell 5: Download the SAM 3.1 checkpoint (gated; HF_TOKEN required).
# `huggingface-cli` is deprecated; the new CLI is `hf` (huggingface_hub >= 1.13).
!hf download facebook/sam3.1 --local-dir models/sam3.1

In [ ]:
# Cell 6: Run the gated tests.
!bash scripts/run_gpu_tests.sh

## Cell 7: Summary

Scroll to the bottom of cell 6's output and read pytest's final summary line
(e.g. `========= 4 passed in 87.3s =========`). That line is the pass/fail signal.